# WFP Syria — data exploration and model starter

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/courses/aabw/notebooks/wfp-syria/starter-data-visualization.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/courses/aabw/notebooks/wfp-syria/starter-data-visualization.ipynb)

**Maintained by the Advanced Analytics for a Better World teaching team**  
AABW teaching notebook · Version 2026.1

Explores the WFP Syria network data and prepares the nominal optimization model.


## How to use this notebook

Work through the marked data-cleaning, nominal-model and robust-model exercises.
The checkpoints explain what is incomplete. A fresh Run all can display the
raw locations, but it does not complete or solve your unfinished models.
Use the companion only when the teaching team releases that worked nominal scaffold.


## Download data and import the required packages


In [ ]:
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'folium': 'folium', 'highspy': 'highspy', 'matplotlib': 'matplotlib', 'networkx': 'networkx', 'numpy': 'numpy', 'pandas': 'pandas', 'pyomo': 'pyomo', 'requests': 'requests', 'seaborn': 'seaborn', 'xlsxwriter': 'xlsxwriter'}
ensure_packages(required_packages)


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import sys

at_colab = "google.colab" in sys.modules
figure_path = "."

# One canonical copy of every dataset and the shared workbook helper.
import hashlib
from urllib.request import urlopen
RESOURCE_FILES = {'Data Set Feedcalculator.xlsx': ('data/cases/feed-calculator.xlsx', '2e3653fe1b3a9995abedfb5902157dea2616c72550b62b7b060442e281bf8fe7'), 'DataSetFeedCalculator.xlsx': ('data/cases/feed-calculator.xlsx', '2e3653fe1b3a9995abedfb5902157dea2616c72550b62b7b060442e281bf8fe7'), 'DataErmeraTimorLeste.xlsx': ('data/cases/timor-leste.xlsx', '31431b07df5b6797630b9ec5023f81aacd76d5bdab4e21cdd3d9fe4e129f021a'), 'DataSyriaCaseWFP.xlsx': ('data/wfp-syria/DataSyriaCaseWFP.xlsx', 'f3e604eb3000d20e0e7eb64c7f847b6e34b83b85a417031a77805b8ea614c55b'), 'WFPCleanWithLocations.xlsx': ('data/wfp-syria/WFPCleanWithLocations.xlsx', 'aa46234010b4b4a94558dab49e0211792ac28bd8a34280dc0927aeca52c0bd8f'), 'WFP_Locations.xlsx': ('data/wfp-syria/WFP_Locations.xlsx', '029fc6ad26c992f0e460954aae9a5d6e1c6bca8a5728c814bd1f233ff54a3ceb'), 'util_AABW.py': ('support/util_AABW.py', '36bce9b87106d200ba5f8c34da6e3990550c6db8045b70041075cb4d1fcef820')}

def fetch_course_file(filename):
    relative, expected = RESOURCE_FILES[filename]
    path = Path(filename)
    if not path.is_file():
        local = Path(relative)
        if local.is_file():
            payload = local.read_bytes()
        else:
            url = 'https://raw.githubusercontent.com/gromicho/teaching/main/' + relative
            with urlopen(url, timeout=45) as response:
                payload = response.read()
        if hashlib.sha256(payload).hexdigest() != expected:
            raise ValueError('Unexpected teaching resource: ' + filename)
        path.write_bytes(payload)
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError('Unexpected local teaching resource: ' + filename)
    return path

for filename in ("util_AABW.py", "DataSyriaCaseWFP.xlsx", "WFP_Locations.xlsx"):
    fetch_course_file(filename)

import util_AABW as abw
import pandas as pd
import numpy as np
import difflib
import seaborn as sns
import matplotlib.pyplot as plt
import pprint
import pyomo.environ as pyo

pp = pprint.PrettyPrinter(indent=2, width=128, compact=True, sort_dicts=True)


# Prepare the data

In [ ]:
data_file = 'DataSyriaCaseWFP.xlsx'

data = abw.ReadWorkbookIntoNamedTuple( data_file )
coordinates = abw.ReadWorkbookIntoNamedTuple( 'WFP_Locations.xlsx' ).Sheet1

### COORDINATE DATA ####
#vary a little between suppliers, transshipmentnodes and demand nodes to make the differences clear when plotting
d_correction = {' D': 0.02,
                ' TS': -0.02}

for n, c in zip(d_correction.keys(), d_correction.values()):
  b = coordinates.NameType.str.contains(n)
  coordinates.loc[b, 'latitude'] = coordinates.loc[b, 'latitude'] + c
  coordinates.loc[b, 'longitude']= coordinates.loc[b, 'longitude']+ abs(c)

data._fields

In [ ]:
# This raw location preview does not depend on answers to the cleaning exercises.
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(coordinates['longitude'], coordinates['latitude'])
ax.set(xlabel='Longitude', ylabel='Latitude', title='Locations before checking network consistency')
plt.show()


Interpretations of NodesTypes Types:
* I = International supplier
* R = Regional supplier
* L = Local market (both supply and deliver)
* D = Delivery point
* TS = Transshipment node

## Clean the data

The issues we deal with with this data are:
- Node names do not always match:
  * some of the node names listed in columns A and B of the Edges - Cost sheet do not math the names on the Nodes - Types sheet
  * not all nodes are included in the markets
  * the names listed in columns of FoodCost are not the same as in NodeTypes
- Food names do not always match: discrepancies between the food names in the nutritional table and in the international price table.
  * Some food names in FoodNutritionalValue and FoodInternationalPrice differ
  * Nutrients in the NutrientRequirements that can not be found in FoodNutritionalValue
- Coordinates are missing from the NodesTypes dataset


## Showing the problems

#### Node and edge names differ sometimes

In [ ]:
set_nodes = set(data.NodesTypes.Name)
set_start_nodes = set(data.EdgesCost.A)
set_end_nodes = set(data.EdgesCost.B)
set_nodes_in_edges = set_start_nodes | set_end_nodes

pd.DataFrame.from_dict(
    { 'In nodes but not in edges' : sorted(set_nodes-set_nodes_in_edges),
      'In edges but not in nodes' : sorted(set_nodes_in_edges-set_nodes) },
    orient='columns'
)

#### Market names differ from node names

In [ ]:
set_markets = set(data.FoodCost.adm1_name)
set_node_names = set(' '.join(n.split(' ')[:-1]) for n in set_nodes)

pd.DataFrame.from_dict(
    { 'In nodes but not in markets' : dict(zip(range(1000),sorted(set_node_names-set_markets))),
      'In markets but not in nodes' : dict(zip(range(1000),sorted(set_markets-set_node_names))),
      'In markets and in nodes' : dict(zip(range(1000),sorted(set_markets&set_node_names))) },
    orient='columns'
).fillna('')

#### Differences FoodNutritionalValue names and FoodInternationalPrice names

In [ ]:
pd.DataFrame.from_dict(
    { 'In nutritional but not in price' :
        sorted(set(data.FoodNutritionalValue.Food) -
               set(data.FoodInternationalPrice.Food)),
      'In price but not in nutritional' :
        sorted(set(data.FoodInternationalPrice.Food) -
               set(data.FoodNutritionalValue.Food)) },
    orient='columns'
)

#### Nutrients in the NutrientRequirements that can not be found in FoodNutritionalValue


In [ ]:
nutrient_requirements = data.NutrientRequirements.T.drop(labels='Type',axis=0)[0].to_dict()

{ n : nutrient_requirements[n] for n in
  nutrient_requirements.keys() - set(data.FoodNutritionalValue.columns) }

## Clean the data

#### Universalize the names of the nodes

In [ ]:
# TODO: Create consistent node names and edge endpoints.


In [ ]:
# TODO: Match each local price record to its supplier node.


#### Universalize the names of the foods

In [ ]:
# TODO: Make food names consistent across the price and nutrient tables.


## Set indexes

In [ ]:
# TODO: Set the node, arc and food indexes; retain columns with drop=False.


#### Add coordinates to the data

In [ ]:
# TODO: Attach a coordinate pair to every node.


In [ ]:
preparation_issues = []
for name, frame, expected in [
    ('NodesTypes', data.NodesTypes, {'NameType', 'coordinates'}),
    ('EdgesCost', data.EdgesCost, {'NameA', 'NameB'}),
    ('FoodCost', data.FoodCost, {'NodeName', 'FoodName'}),
    ('FoodInternationalPrice', data.FoodInternationalPrice, {'FoodName'}),
]:
    missing = expected - set(frame.columns)
    if missing:
        preparation_issues.append(f'{name}: create columns {sorted(missing)}.')
if data.NodesTypes.index.name != 'NameType':
    preparation_issues.append('Set the node index to NameType.')
if list(data.EdgesCost.index.names) != ['NameA', 'NameB']:
    preparation_issues.append('Set the arc index to NameA, NameB.')
if data.FoodNutritionalValue.index.name != 'Food':
    preparation_issues.append('Set the nutritional table index to Food.')
if not preparation_issues:
    if not set(data.EdgesCost.index.get_level_values(0)).union(
            data.EdgesCost.index.get_level_values(1)).issubset(data.NodesTypes.index):
        preparation_issues.append('Every arc endpoint must name a known node.')
    if data.NodesTypes['coordinates'].isna().any():
        preparation_issues.append('Some nodes still have no coordinate pair.')
preparation_ready = not preparation_issues
for issue in preparation_issues:
    print(issue)
print('Preparation checkpoint:', 'ready for modelling' if preparation_ready else 'still incomplete')


# Visualize the network

In [ ]:
# set the abbreviations, colors and legend names
abbreviation = {
    'Aleppo' : 'Al',
    'Amman' : 'Am',
    'Ar Raqqa' : 'AR',
    'As_Suweida' : 'AS',
    'Beirut' : 'Be',
    'Damascus' : 'Dm',
    'Daraa' : 'Da',
    'Dayr_Az_Zor' : 'DZ',
    'Gaziantep' : 'Gz',
    'Hama' : 'Hm',
    'Hassakeh' : 'Hs',
    'Homs' : 'Ho',
    'Idleb' : 'Id',
    'Jubb_al_Jarrah' : 'JJ',
    'Qamishli' : 'Qi'
}

category_colors = dict(
    I= 'pink',
    R= 'lightgreen',
    L= 'orange',
    D= 'red',
    TS= 'lightblue'
  )

node_type = dict(
    I= 'International supplier',
    R= 'Regional supplier',
    L= 'Local market (both supply and deliver)',
    D= 'Delivery point',
    TS= 'Transshipment node'
)

#### Using Folium

In [ ]:
if preparation_ready:
    import folium as fl
    import folium.plugins
    import pandas as pd

    def add_markers_from_dict(Map, df, d_col = category_colors):

      for n,c,t in zip(df.index, df.coordinates, df.Type) :
        fl.Marker(c,
                  icon=fl.plugins.BeautifyIcon(icon='',
                                               icon_shape='circle',
                                               background_color=d_col[t],
                                               border_width=0,
                                               inner_icon_style='font-size:15px; text-align:center'),
                  tooltip=n).add_to(Map)

      d = df.coordinates.to_dict()
      for a,b in data.EdgesCost.index:
        fl.PolyLine([d[a], d[b]], color="black", weight=2.5, opacity=1).add_to(Map)
      return Map

    def add_legend(d_col = category_colors, node_type = node_type):
      legend_html = """
      <div style="position: fixed; bottom: 160px; left: 80px; z-index:9999; font-size: 22px;">
      <p><strong>Legend</strong></p>
      """
      for category, color in d_col.items():
          legend_html += f'<i class="fa fa-circle fa-1x" style="color:{color}"></i> {node_type[category]}<br>'
      legend_html += "</div>"
      return legend_html

    # c = geopy.Nominatim(user_agent='user_agent').geocode('Syria')
    c = (coordinates.latitude.mean(), coordinates.longitude.mean())
    Map = fl.Map(location=c, zoom_start=7)
    Map = add_markers_from_dict(Map, data.NodesTypes)
    legend_html = add_legend()
    Map.get_root().html.add_child(folium.Element(legend_html))
    Map
else:
    print("Visualization skipped: complete the data-preparation steps first.")


#### Using Networkx

In [ ]:
if preparation_ready:
    import networkx as nx
    import matplotlib.patches as mpatches

    def create_graph(data):
      # Create graph
      g = nx.DiGraph()

      for node, row in data.NodesTypes.iterrows():
          g.add_node(node,**row.to_dict())
      for edge, row in data.EdgesCost.iterrows():
          g.add_edge(*edge,**row.to_dict())
      return g

    g = create_graph( data)

    #create essential functions and names
    pure_name = lambda n : ' '.join( n.split(' ')[:-1] )
    type_from_name = lambda n : n.split(' ')[-1]

    #create information for the legend
    node_labels = { n : abbreviation[pure_name(n)] for n in data.NodesTypes.index }
    node_colors = [ category_colors[t] for t in data.NodesTypes.Type ]

    #create positions for the nodes
    pos = dict()
    for name, (lat, lon) in zip(data.NodesTypes.index, data.NodesTypes.coordinates):
        pos[name] = [lon,lat]

    pos = nx.spring_layout(g, pos=pos,
                           k=13, scale=8, weight='tCost',
                           iterations=2, seed=2023)

    #plot the figure
    plt.figure(figsize=(10,6))

    nx.draw(g, pos, with_labels=False, node_size=500, node_color=node_colors)
    nx.draw_networkx_labels(g, pos, labels=node_labels)
    nx.draw_networkx_edges(g, pos,
    width=[1e-5+5e-3*c for c in nx.get_edge_attributes(g,'tCost').values()],
    edge_color='black'
    )

    legend_elements = [ mpatches.Patch(color=color, label=node_type[category])
        for category, color in category_colors.items() ]

    # Add the legend to the plot
    plt.legend(handles=legend_elements, loc='lower right')

    # Export explicitly with plt.savefig(...) when a figure file is needed.

    plt.show()

else:
    print("Visualization skipped: complete the data-preparation steps first.")


# Implement the nominal model

## Nominal supply-chain model and assumptions

Let $N_S,N_T,N_B$ denote suppliers, transshipment nodes and beneficiaries,
$E$ the directed arcs, $K$ commodities and $L$ the nutrients with measured
coefficients. Let $F_{ijk}\geq0$ be commodity flow on an arc and $R_k\geq0$
the common ration per beneficiary. The model is

$$\min\ \sum_{(i,j)\in E,\,i\in N_S}\sum_{k\in K}pc_{ik}F_{ijk}
 +\sum_{(i,j)\in E}\sum_{k\in K}(tc_{ij}+hc_j)F_{ijk}$$

$$\sum_{i:(i,j)\in E}F_{ijk}=\sum_{i:(j,i)\in E}F_{jik}
\quad(j\in N_T,\ k\in K),$$

$$\sum_{i:(i,j)\in E}F_{ijk}\geq dem_jR_k
\quad(j\in N_B,\ k\in K),$$

$$\sum_{k\in K}\overline v_{kl}R_k\geq r_l\quad(l\in L).$$

Here `dem` is the number of beneficiaries, `pc` the mean procurement price,
`tc` the supplied transport cost, and $\overline v,r$ the nutritional values
and requirements. Use consistent commodity, nutrient and monetary units when
interpreting the workbook. A supplier/commodity combination without a recorded
price is unavailable: its outgoing flow must be zero, rather than merely
penalized by an enormous invented price.

**Two explicit limitations of this dataset:** the workbook supplies no handling
cost table, so this exercise sets $hc_j=0$. The handling term is still included
in the model. Iodine has a requirement but no commodity coefficients; it cannot
be checked with these data and is excluded from $L$. This does not establish
iodine adequacy. Explain both assumptions when reporting a solution.

Conservation is imposed separately for every commodity, so a transshipment
node cannot turn one food into another. A completely isolated transshipment
node has the trivial equation $0=0$.


In [ ]:
def WFP_model(data):
    """Build and return the nominal WFP model."""
    # TODO: implement the sets, parameters, variables, objective, and constraints.
    raise NotImplementedError("Complete the nominal WFP model before solving it.")


In [ ]:
m = None
if preparation_ready:
    try:
        m = WFP_model(data)
    except NotImplementedError as error:
        print(error)
else:
    print('Complete the preparation checkpoint before building the nominal model.')


In [ ]:
if m is not None:
    from teaching_utils import solve_checked
    results = solve_checked(m, 'appsi_highs')
else:
    print('Nominal solve pending: implement the model first.')


In [ ]:
if m is not None:
    fin_F = {(i,j,k): pyo.value(m.F[i,j,k]) for i,j in m.E for k in m.K if  pyo.value(m.F[i,j,k])>0}
    fin_R = {k: pyo.value(m.R[k]) for k in m.K if pyo.value(m.R[k])>0}
else:
    print('Nominal results are not available before completing the model.')


In [ ]:
if m is not None:
    print('Costs of designing the supply chain like this: $' + str(round(pyo.value(m.Cost),2)))
else:
    print('Nominal results are not available before completing the model.')


### Visualize the results
can plot all different commodity supply chains together (but the colors overlap), or dan show them separately. To plot them all, pass along the argument 'all', otherwise choose one present in `fin_R.keys()`.

In [ ]:
import folium as fl
import pandas as pd


## Exercise: uncertainty in nutritional values

The course assignment keeps the mean procurement prices and asks us to protect
the **nutritional constraints** against uncertain commodity composition.
For each nutrient $l$, write its coefficient vector as
$v_l=\overline v_l+\sigma_l z_l$, where

$$\mathcal U_l=\{z_l:\|z_l\|_2\leq\rho\}.$$

The requirement is now

$$\sum_{k\in K}(\overline v_{kl}+\sigma_l z_{kl})R_k\geq r_l
\quad\text{for every }z_l\in\mathcal U_l,\quad l\in L.$$

**Your tasks:** derive a deterministic counterpart, implement it in
`robust_WFP_model`, and compare several values of $\rho$. Start with equal
uncertainty scales, $\sigma_l=1$ in each nutrient's stated units, as the teaching
exercise assumes. These scales are an explicit modelling choice, not variances
estimated from the price history. Nutrients use different physical units, so a
real application needs justified nutrient-specific scales.

Keep the nominal objective and network constraints. At $\rho=0$, your model
must recover the nominal problem. For larger radii the feasible set becomes
smaller, so its optimal minimum cost cannot decrease when both problems are
feasible and solved to optimality. Infeasibility is a possible result, not an
excuse to read missing variable values.

If you express a norm with squared variables, state the required sign
restrictions on any auxiliary variable. Install the conic-capable solver at the
stage below; use the course's licensed runtime for the full robust model.
The completed robust implementation is supplied separately by the teaching team.


In [ ]:
def robust_WFP_model(data, rho):
    """Build and return the robust WFP model for radius ``rho``."""
    # TODO: extend the nominal formulation with the robust counterpart.
    raise NotImplementedError("Complete the robust WFP model before solving it.")


In [ ]:
m_rob = None
if preparation_ready:
    try:
        m_rob = robust_WFP_model(data, 2)
    except NotImplementedError as error:
        print(error)
else:
    print('Complete the preparation checkpoint before building the robust model.')


In [ ]:
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'gurobipy': 'gurobipy'}
ensure_packages(required_packages)


In [ ]:
if m_rob is not None:
    from teaching_utils import solve_checked
    results = solve_checked(m_rob, 'gurobi_direct')
else:
    print('Robust solve pending: implement the robust model first.')


In [ ]:
if m_rob is not None:
    rob_F = {(i,j,k): pyo.value(m_rob.F[i,j,k]) for i,j in m_rob.E for k in m_rob.K if  pyo.value(m_rob.F[i,j,k])>0}
    rob_R = {k: pyo.value(m_rob.R[k]) for k in m_rob.K if pyo.value(m_rob.R[k])>0}
else:
    print('Robust results are not available before completing the model.')


### Analyze the results for different values of $\rho$

In [ ]:
rho_values = [0, 0.5, 1, 1.5, 2]
# TODO: after implementing robust_WFP_model, solve the model for every value
# in rho_values and collect total food, number of ingredients, and total cost.
rho_values


### Visualize the results

### See how your constraints turned out